# **README**

**Please make a copy of this colab before executing the cells.**

This colab is to demonstrate A2A and Agent Engine integration and registration on Gemini Enterprise.

**Assumptions / Limitations**

* The integration is currently in preview and does not support all the
capabilities in A2A, specifically the streaming capabilities. Streaming is explicitly turned off in the examples below even if the agent card has streaming set to true.

* **This colab assumes that the authenticated user has Vertex AI User role.

      

## Prerequisites

### Authorize the colab

In [ ]:
# Auth colab
from google.colab import auth
auth.authenticate_user()

### install prerequisites

In [ ]:
!
!pip install "google-cloud-aiplatform[agent_engines, adk]" --force-reinstall --quiet
!pip install "a2a-sdk >= 0.3.5" --force-reinstall --quiet
!pip install cryptography==42.0.8 --force-reinstall --quiet
!pip install starlette --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.1/223.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.5/229.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip show google-cloud-aiplatform

Name: google-cloud-aiplatform
Version: 1.126.1
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, google-genai, packaging, proto-plus, protobuf, pydantic, shapely, typing_extensions
Required-by: google-adk


### define helper functions

In [ ]:
from a2a.types import MessageSendParams,Message, Role, Part, TextPart
from starlette.requests import Request
from starlette.datastructures import Headers, URL
import asyncio
from typing import Any
import json
import uuid

def receive_wrapper(data):
  async def receive():
      byte_data = json.dumps(data).encode("utf-8")
      return {"type": "http.request", "body": byte_data, "more_body": False}
  return receive


def build_post_request(data: dict[str, Any] = None, path_params: dict[str, str] = None) -> Request:
  scope = {
    "type": "http",
    "http_version": "1.1",
    "headers": [(b"content-type", b"application/json")],
    "app": None,
  }
  if(path_params):
    scope['path_params'] = path_params
  receiver = receive_wrapper(data)
  return Request(scope, receiver)

def build_get_request(path_params: dict[str, str]) -> Request:
  scope = {
    "type": "http",
    "http_version": "1.1",
    'query_string': b'',
    "app": None,
  }
  if(path_params):
    scope['path_params'] = path_params

  async def receive():
    return {"type": "http.disconnect"}
  #receiver = receive_wrapper(data)
  return Request(scope, receive)

### init genAI sdk

In [ ]:
import vertexai
from google.genai import types
# Replace these as appropriate
PROJECT_ID = "14326790950" # @param{type:"string"}
# 429499802777
# 14326790950 - bap-labs-agent-demo
LOCATION = "us-central1" # @param{type:"string"}
STORAGE = "a2a_ae_storage_stage" # @param{type:"string"}
# a2a_ae_fishfood_storage, a2a_ae_storage_stage
API_ENDPOINT = f"{LOCATION}-aiplatform.sandbox.googleapis.com"

# Using autopush for fishfood
# this is required for general init
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    api_endpoint=API_ENDPOINT,
    staging_bucket="gs://" + STORAGE,
    )


# this is required to create a genAI client
client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        api_version="v1beta1",
    ),
)

## Part1. Create & deploy agent

---



* CUJ1: define agent components
* CUJ2: create the local agent with a2aTemplate
* CUJ3: test out the local agent
* CUJ4: deploy agent to agent engine
* CUJ5: test remote agent via SDK methods


additionally
*  create the agentCard with a2aTemplate utils
*  CUJ1 - 7 uses inMemoryTaskStore by default
*  we limit the cloud run instance and worker to 1 to ensure inMemoryTaskStore works for fishfooding purpose




### Define agent components


1. agent card
2. agent executor: internally invokes ADK Agent

Agent Card


---

The agent card below is created using the utility method in A2a Agent Engine template. The utility builds the card based on MVP integration. The following are to be noted.

1. Streaming is turned off
2. Supports Authenticated Extended Card is turned on.

create_agent_card has conditionally optional parameters

agent_name (Optional[str]): The name of the agent.
description (Optional[str]): A description of the agent.
skills (Optional[List[AgentSkill]]): A list of AgentSkills.
agent_card (Optional[Dict[str, Any]]): Agent Card as a dictionary.

Either the complete agent card is supplied as agent_Card dictionary or the three parameters agent_name, description and skills are expected to be entered.

If an Agent card is supplied as a parameter, validation errors might show depending on whether the card hits the current integration limitations.

In [ ]:
from a2a.types import AgentCard, AgentSkill, AgentCapabilities, TransportProtocol
from vertexai.preview.reasoning_engines.templates.a2a import create_agent_card

agent_skill = AgentSkill(
        id='question_answer',
        name='Q&A Agent',
        description='A helpful assistant agent that can answer questions.',
        tags=['Question-Answer'],
        examples=[
            'Who is leading 2025 F1 Standings?',
            'Where can i find an active volcano?',
        ],
    )
qna_agent_card = create_agent_card(agent_name='Q&A Agent',
        description='A helpful assistant agent that can answer questions.',
        skills=[agent_skill])


In [ ]:
qna_agent_card

AgentCard(additional_interfaces=None, capabilities=AgentCapabilities(extensions=None, push_notifications=None, state_transition_history=None, streaming=False), default_input_modes=['text/plain'], default_output_modes=['application/json'], description='A helpful assistant agent that can answer questions.', documentation_url=None, icon_url=None, name='Q&A Agent', preferred_transport='HTTP+JSON', protocol_version='0.3.0', provider=None, security=None, security_schemes=None, signatures=None, skills=[AgentSkill(description='A helpful assistant agent that can answer questions.', examples=['Who is leading 2025 F1 Standings?', 'Where can i find an active volcano?'], id='question_answer', input_modes=None, name='Q&A Agent', output_modes=None, security=None, tags=['Question-Answer'])], supports_authenticated_extended_card=True, url='http://localhost:9999/', version='1.0.0')

In [ ]:
# agent executor
"""
An ADK agent that uses google search to answer questions.
"""
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.tasks import TaskUpdater
from a2a.types import TaskState, TextPart, UnsupportedOperationError
from a2a.utils import new_agent_text_message
from a2a.utils.errors import ServerError
from google.adk import Runner
from google.adk.agents import LlmAgent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search_tool
from google.genai import types

class QnAAgentExecutor(AgentExecutor):
    """Agent executor that uses the ADK to answer questions."""

    def __init__(self):
        self.agent = None
        self.runner = None

    def _init_agent(self):
        self.agent = LlmAgent(
            model='gemini-2.0-flash-exp',
            name='question_answer_agent',
            description='A helpful assistant agent that can answer questions.',
            instruction="""Respond to the query using google search""",
            tools=[google_search_tool.google_search],
        )
        self.runner = self.runner = Runner(
            app_name=self.agent.name,
            agent=self.agent,
            artifact_service=InMemoryArtifactService(),
            session_service=InMemorySessionService(),
            memory_service=InMemoryMemoryService(),
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        raise ServerError(error=UnsupportedOperationError())

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        if self.agent is None:
            self._init_agent()

        query = context.get_user_input()

        updater = TaskUpdater(event_queue, context.task_id, context.context_id)

        if not context.current_task:
            await updater.submit()

        await updater.start_work()

        content = types.Content(role='user', parts=[types.Part(text=query)])
        session = await self.runner.session_service.get_session(
            app_name=self.runner.app_name,
            user_id='123',
            session_id=context.context_id,
        ) or await self.runner.session_service.create_session(
            app_name=self.runner.app_name,
            user_id='123',
            session_id=context.context_id,
        )

        async for event in self.runner.run_async(
            session_id=session.id, user_id='123', new_message=content
        ):
            if event.is_final_response():
                parts = event.content.parts
                text_parts = [
                    TextPart(text=part.text) for part in parts if part.text
                ]
                await updater.add_artifact(
                    text_parts,
                    name='result',
                )
                await updater.complete()
                break
            await updater.update_status(
                TaskState.working, message=new_agent_text_message('Working...')
            )
        else:
            await updater.update_status(
                TaskState.failed,
                message=new_agent_text_message(
                    'Failed to generate a response.'
                ),
            )

### Create a local agent

In [ ]:
from vertexai.preview.reasoning_engines import A2aAgent

a2a_agent = A2aAgent(agent_card=qna_agent_card, agent_executor_builder=QnAAgentExecutor)
a2a_agent.set_up()

# you may see error on supports_authenticated_extended_card which is due to limitation of MVP. please ignore.
# Using in progress a2a-sdk, this issue will be addressed in the a2a sdk.
# https://github.com/a2aproject/a2a-python/issues/438


### Test the local agent

#### v1/card

In [ ]:
request = build_get_request(None)
response = await a2a_agent.handle_authenticated_agent_card(request=request, context=None)
print(response)

{'capabilities': {'streaming': False}, 'defaultInputModes': ['text'], 'defaultOutputModes': ['text'], 'description': 'A helpful assistant agent that can answer questions.', 'name': 'Q&A Agent', 'preferredTransport': 'HTTP+JSON', 'protocolVersion': '0.3.0', 'skills': [{'description': 'A helpful assistant agent that can answer questions.', 'examples': ['Who is leading 2025 F1 Standings?', 'Where can i find an active volcano?'], 'id': 'question_answer', 'name': 'Q&A Agent', 'tags': ['Question-Answer']}], 'supportsAuthenticatedExtendedCard': True, 'url': 'https://us-central1-aiplatform.googleapis.com/v1beta1/projects/14326790950/locations/us-central1/reasoningEngines/test-agent-engine/a2a', 'version': '1.0.0'}


### Deploy the agent to agent engine

In [ ]:
config={"staging_bucket": "gs://" + STORAGE,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "a2a-sdk >= 0.3.4",
            # "google-cloud-alloydb-connector[asyncpg]",
        ],
        "http_options": {
            "api_version": "v1beta1",
        },
        # needed to run with inMemoryTaskStore
        "max_instances" : 1, # one cloud run instance
        "env_vars" : {
        "NUM_WORKERS": "1", # one process
    },
  }

remote_agent = client.agent_engines.create(
        agent=a2a_agent,
        config=config)
print(remote_agent)


INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.128.0', 'a2a-sdk': '0.3.15', 'pydantic': '2.12.4', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.4'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'a2a-sdk >= 0.3.4', 'cloudpickle==3.1.2', 'pydantic==2.12.4']
INFO:vertexai_genai.agentengines:Using bucket a2a_ae_storage_stage
INFO:vertexai_genai.agentengines:Wrote to gs://a2a_ae_storage_stage/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://a2a_ae_storage_stage/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://a2a_ae_storage_stage/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:The provided agent framework None is not supported. 

api_client=<vertexai._genai.agent_engines.AgentEngines object at 0x7e358f66bce0> api_async_client=<vertexai._genai.agent_engines.AsyncAgentEngines object at 0x7e358f66ba70> api_resource=ReasoningEngine(
  name='projects/14326790950/locations/us-central1/reasoningEngines/6346570231523049472',
  spec=ReasoningEngineSpec(
    agent_framework='custom',
    class_methods=[
      {
        'a2a_agent_card': '{"additionalInterfaces":null,"capabilities":{"extensions":null,"pushNotifications":null,"stateTransitionHistory":null,"streaming":false},"defaultInputModes":["text"],"defaultOutputModes":["text"],"description":"A helpful assistant agent that can answer questions.","documentationUrl":null,"iconUrl":null,"name":"Q&A Agent","preferredTransport":"HTTP+JSON","protocolVersion":"0.3.0","provider":null,"security":null,"securitySchemes":null,"signatures":null,"skills":[{"description":"A helpful assistant agent that can answer questions.","examples":["Who is leading 2025 F1 Standings?","Where can 

In [ ]:
# @title Get the remote agent engine.

AGENT_ENGINE_RESOURCE = 'projects/14326790950/locations/us-central1/reasoningEngines/6346570231523049472' # @param{type:"string"}

if not AGENT_ENGINE_RESOURCE:
  full_resource_name = remote_agent.api_resource.name
  REASONING_ENGINE_ID = full_resource_name.split('/')[-1]
  # '9072391498175086592'
  print(f"Successfully extracted Reasoning Engine ID: {REASONING_ENGINE_ID}")
  AGENT_ENGINE_RESOURCE = f"projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{REASONING_ENGINE_ID}"
  print(f"Constructed full resource path: {AGENT_ENGINE_RESOURCE}")

remote_agent = client.agent_engines.get(
    name=AGENT_ENGINE_RESOURCE,
)
remote_agent


AgentEngine(api_resource.name='projects/14326790950/locations/us-central1/reasoningEngines/6346570231523049472')

### Test Remote Agent via SDK methods

#### v1/message:send

In [ ]:
# Create a message
message_data = {
  "messageId": "someid",
  "role": "user",
  "parts": [{"kind": "text", "text": "Who is leading the current F1 standings"}],
}
# Invoke the agent
response = await remote_agent.on_message_send(**message_data,)
print(response)


# initial_task, _ = await anext(response)

# # 4. Now you have the initial Task object. You can inspect it
# #    and extract the task_id for future use.
# print("--- Initial Task Object from send_message Response ---")
# print("----------------------------------------------------")

# if initial_task:
#     task_id = initial_task.id
#     print(f"\nThe Task ID is: {task_id}")


[(Task(artifacts=[Artifact(artifact_id='e6caf69a-185e-40b9-b66d-c98f076e4b88', description='', extensions=None, metadata={}, name='result', parts=[Part(root=TextPart(kind='text', metadata=None, text="As of November 19, 2025, the leader in the F1 World Drivers' Championship standings is Lando Norris with 390 points. He drives for McLaren.\n"))])], context_id='28a27bd3-c725-4d94-9401-395c67d6de34', history=[Message(context_id='28a27bd3-c725-4d94-9401-395c67d6de34', extensions=None, kind='message', message_id='someid', metadata={}, parts=[Part(root=TextPart(kind='text', metadata=None, text='Who is leading the current F1 standings'))], reference_task_ids=None, role=<Role.user: 'user'>, task_id='c4ff8fde-da3e-4dc3-b628-d1455deaa6fb')], id='c4ff8fde-da3e-4dc3-b628-d1455deaa6fb', kind='task', metadata=None, status=TaskStatus(message=Message(context_id=None, extensions=None, kind='message', message_id='', metadata={}, parts=[], reference_task_ids=None, role=<Role.agent: 'agent'>, task_id=None)

#### v1/card

In [ ]:
remote_agent_card = await remote_agent.handle_authenticated_agent_card()
print(remote_agent_card)

additional_interfaces=None capabilities=AgentCapabilities(extensions=None, push_notifications=None, state_transition_history=None, streaming=False) default_input_modes=['text'] default_output_modes=['text'] description='A helpful assistant agent that can answer questions.' documentation_url=None icon_url=None name='Q&A Agent' preferred_transport='HTTP+JSON' protocol_version='0.3.0' provider=None security=None security_schemes=None signatures=None skills=[AgentSkill(description='A helpful assistant agent that can answer questions.', examples=['Who is leading 2025 F1 Standings?', 'Where can i find an active volcano?'], id='question_answer', input_modes=None, name='Q&A Agent', output_modes=None, security=None, tags=['Question-Answer'])] supports_authenticated_extended_card=True url='https://us-central1-aiplatform.googleapis.com/v1beta1/projects/14326790950/locations/us-central1/reasoningEngines/6346570231523049472/a2a' version='1.0.0'


## Part 2: Call via Http, A2A Clients

### HTTP

In [ ]:
# Helper for the http based test
from google.auth import default
from google.auth.transport.requests import Request
import httpx

def get_bearer_token():
  try:
    credentials, project = default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    request = Request()
    credentials.refresh(request)
    return credentials.token
  except Exception as e:
    print(f"Error getting credentials: {e}")
    print("Please ensure you have authenticated with 'gcloud auth application-default login'.")
  return None

import httpx
import json

bearer_token = get_bearer_token()
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json"
}

In [ ]:
# construct a2a endpoint.
# v1beta1 is needed for public review
API_ENDPOINT = f"{LOCATION}-aiplatform.googleapis.com"
A2A_ENDPOINT=f"https://{API_ENDPOINT}/v1beta1/{AGENT_ENGINE_RESOURCE}/a2a"

# https://us-central1-autopush-aiplatform.sandbox.googleapis.com/v1beta1/projects/429499802777/locations/us-central1/reasoningEngines/9117568231937146880

#### v1/message:send

In [ ]:
message_send_end_point = f"{A2A_ENDPOINT}/v1/message:send"
print(message_send_end_point)
message_data = {
  "message": {
    "messageId": "bde6f754-f88b-4e58-98fc-09239a55925e",
    "role": "1",
    "content": [
      {
        "text": "Who is leading the current F1 drivers standings"
      }
    ]
  },
  "metadata": {
    "source": "curl_client",
    "user_agent": "test_script"
  }
}
try:
  response = httpx.post(message_send_end_point, json=message_data, headers=headers)
  response.raise_for_status()
  print(json.dumps(response.json(), indent=4))
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e}")
    print(f"Response body: {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to send the request: {e}")


task_id = response.json()['task']['id']
print(f"The Task ID is: {task_id}")


https://us-central1-aiplatform.googleapis.com/v1beta1/projects/14326790950/locations/us-central1/reasoningEngines/6346570231523049472/a2a/v1/message:send
{
    "task": {
        "contextId": "c16da1d6-16b9-4afd-920d-1106227f8122",
        "id": "4d6441ba-4278-4a6c-bff6-4d8d23b4f65f",
        "history": [
            {
                "contextId": "c16da1d6-16b9-4afd-920d-1106227f8122",
                "messageId": "bde6f754-f88b-4e58-98fc-09239a55925e",
                "metadata": {},
                "taskId": "4d6441ba-4278-4a6c-bff6-4d8d23b4f65f",
                "role": "ROLE_USER",
                "content": [
                    {
                        "text": "Who is leading the current F1 drivers standings"
                    }
                ]
            }
        ],
        "status": {
            "state": "TASK_STATE_SUBMITTED"
        }
    }
}
The Task ID is: 4d6441ba-4278-4a6c-bff6-4d8d23b4f65f


#### v1/tasks/{taskid}

#### v1/card

In [ ]:
card_endpoint = f"{A2A_ENDPOINT}/v1/card"
try:
  response = httpx.get(card_endpoint, headers=headers)
  response.raise_for_status()
  a2a_agent_card = json.dumps(response.json(), indent=4)
  print(a2a_agent_card)
except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e}")
    print(f"Response body: {e.response.text}")
except httpx.RequestError as e:
    print(f"An error occurred while trying to send the request: {e}")


{
    "description": "A helpful assistant agent that can answer questions.",
    "capabilities": {
        "streaming": false
    },
    "defaultOutputModes": [
        "text"
    ],
    "preferredTransport": "HTTP+JSON",
    "defaultInputModes": [
        "text"
    ],
    "skills": [
        {
            "description": "A helpful assistant agent that can answer questions.",
            "examples": [
                "Who is leading 2025 F1 Standings?",
                "Where can i find an active volcano?"
            ],
            "tags": [
                "Question-Answer"
            ],
            "name": "Q&A Agent",
            "id": "question_answer"
        }
    ],
    "protocolVersion": "0.3.0",
    "version": "1.0.0",
    "name": "Q&A Agent",
    "supportsAuthenticatedExtendedCard": true,
    "url": "https://us-central1-aiplatform.googleapis.com/v1beta1/projects/14326790950/locations/us-central1/reasoningEngines/6346570231523049472/a2a"
}


In [ ]:
import json
parsed_card = json.loads(a2a_agent_card)
compact_card = json.dumps(parsed_card)


## Part 3: Register with Gemini Enterprise

In [ ]:
# @title Helper to register agent on gemini enterprise
import os
import requests
def register_agent_on_gemini_enterprise(
    project_number: str,
    app_id: str,
    agent_card: str,
    agent_name: str,
    display_name: str,
    description: str,
    agent_authorization: str = None,
):
    """Register an Agent Engine to Gemini Enterprise.

    Args:
        project_number: GCP project number
        app_id: Gemini Enterprise application ID
        reasoning_engine_resource_name: Full resource name of deployed Agent Engine
        display_name: Display name for the agent in Gemini Enterprise
        description: Description of the agent

    Returns:
        dict: Response from Discovery Engine API
    """
    raw_location = "global"

    # Map region to Discovery Engine location and base URL
    if raw_location.startswith("us-"):
        location = "us"
        base_url = "https://us-discoveryengine.googleapis.com"
    elif raw_location.startswith("eu-"):
        location = "eu"
        base_url = "https://eu-discoveryengine.googleapis.com"
    else:
        location = "global"
        base_url = "https://discoveryengine.googleapis.com"

    #test-discoveryengine.sandbox.googleapis.com
    # Construct API endpoint
    api_endpoint = (
        f"{base_url}/v1alpha/projects/{project_number}/"
        f"locations/{location}/collections/default_collection/engines/{app_id}/"
        f"assistants/default_assistant/agents"
    )

    #explicitly_escaped_string = compact_card.replace('"', '\\\"')
    #explicitly_escaped_string
    # Prepare request payload
    payload = {
        "name": agent_name,
        "displayName": display_name,
        "description": description,
        "a2aAgentDefinition": {
            "jsonAgentCard": agent_card
        },
    }

    if agent_authorization:
      payload["authorization_config"] = {
          "agent_authorization": agent_authorization
      }

    # Get access token
    bearer_token = get_bearer_token()

    # Prepare headers
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Goog-User-Project": project_number,
    }

    # Make POST request
    print(f"Registering agent to Gemini Enterprise...")
    print(f"API Endpoint: {api_endpoint}")
    print(f"Display Name: {display_name}")

    response = requests.post(api_endpoint, headers=headers, json=payload)

    if response.status_code == 200:
        print("✓ Agent registered successfully!")
        return response.json()
    else:
        print(f"✗ Registration failed with status code: {response.status_code}")
        print(f"Response: {response.text}")
        response.raise_for_status()

In [ ]:
# @title Invoke registration api

ge_project_number = '14326790950' # @param{type:"string"}
ge_app_id = 'a2a-ae-demo-1763669359558_1763669368216' # @param{type:"string"}
agent_name = 'WebSearchAgent' # @param{type:"string"}
agent_display_name = 'Web Search Agent' # @param{type:"string"}
agent_description = 'Question and Answer Agent' # @param{type:"string"}
agent_authorization = '' # @param{type:"string"}
#projects/14326790950/locations/global/authorizations/A2A_ON_GE

register_agent_on_gemini_enterprise(
    project_number=ge_project_number,
    app_id=ge_app_id,
    agent_card=compact_card,
    agent_name=agent_name,
    display_name=agent_display_name,
    description=agent_description,
    agent_authorization=agent_authorization,
)

Registering agent to Gemini Enterprise...
API Endpoint: https://discoveryengine.googleapis.com/v1alpha/projects/14326790950/locations/global/collections/default_collection/engines/a2a-ae-demo-1763669359558_1763669368216/assistants/default_assistant/agents
Display Name: Web Search Agent
✓ Agent registered successfully!


{'name': 'projects/14326790950/locations/global/collections/default_collection/engines/a2a-ae-demo-1763669359558_1763669368216/assistants/default_assistant/agents/9054384484344460644',
 'displayName': 'Web Search Agent',
 'description': 'Question and Answer Agent',
 'createTime': '2025-11-20T22:21:49.074840230Z',
 'state': 'ENABLED',
 'a2aAgentDefinition': {'jsonAgentCard': '{"description": "A helpful assistant agent that can answer questions.", "capabilities": {"streaming": false}, "defaultOutputModes": ["text"], "preferredTransport": "HTTP+JSON", "defaultInputModes": ["text"], "skills": [{"description": "A helpful assistant agent that can answer questions.", "examples": ["Who is leading 2025 F1 Standings?", "Where can i find an active volcano?"], "tags": ["Question-Answer"], "name": "Q&A Agent", "id": "question_answer"}], "protocolVersion": "0.3.0", "version": "1.0.0", "name": "Q&A Agent", "supportsAuthenticatedExtendedCard": true, "url": "https://us-central1-aiplatform.googleapis.co